# Lab02 chuẩn — ResNet18 và DenseNet121 trên CIFAR10

Hai kiến trúc, bốn điều kiện: ResNet head/fine-tune và DenseNet head/fine-tune. Khởi tạo pretrained ImageNet; head cache feature, train 10 epoch. Mỗi model có 3 trial Optuna tìm LR chung, tối đa 10 epoch/trial. AdamW, weight decay 1e-4, cosine. Không augmentation/dropout/pruning. Checkpoint/objective/early stopping cùng theo validation loss; fine-tune patience 3, min_delta 0.001, từ epoch 4. Đây là notebook chuẩn mới; không dùng kết quả 5+5 trial LR chung/riêng cũ.

Notebook dùng chung src.standard_suite với local. Từng stage gọi backend riêng, vẽ loss/accuracy/LR ngay mỗi epoch và bảng search sau mỗi trial. Test chỉ sau khi cả hai model đã khóa lựa chọn. Không cam kết dưới 5 giờ trước khi đo.

## Pipeline Execution Roadmap

| Bước | Phase | Nội dung | Module | Artifact |
|---|---|---|---|---|
| 1 | Setup | Source, GPU, dependencies, seed | bootstrap; experiment.seed_all | setup/environment.json |
| 2 | Dữ liệu/EDA | Official CIFAR10, checksum, split, bảng và ảnh | eda.build, read_raw, locked_split | outputs/eda/* |
| 3 | Preprocess | RGB 224x224 theo weights, không augmentation | data.image_transform | actual_model_preprocessing.png, preprocessing.json |
| 4 | Architecture | Hai model; frozen/trainable | models.configure_trainable | architecture.txt, parameter_scope.csv |
| 5 | ResNet head | Cache train/val feature một lần; train classifier | Suite.head, cached_features | feature cache, history, head_selected/best.pt |
| 6 | ResNet tuning | 3 LR trial, mở layer4+fc | Suite.search; experiment.fit | study.sqlite3, trials.csv, trial_XX/*, selection.json |
| 7 | DenseNet head | Cache feature và head mới | Suite.head | như ResNet head |
| 8 | DenseNet tuning | 3 LR trial, block4+norm5+classifier | Suite.search | study/trial artifacts |
| 9 | Test sau khóa | Bốn điều kiện, confusion/ROC/PR/predictions | Suite.evaluate, reporting.export_evaluation | evaluation/head và evaluation/finetune |
| 10 | Save/load | Model độc lập, full test, probability tolerance | Suite.evaluate | loaded/, roundtrip.json |
| 11 | Tổng hợp/bàn giao | So sánh bốn điều kiện, manifest và ZIP | notebook; Suite.manifest | comparison.csv/png, registry, ZIP |

Các checkpoint trước hoàn tất được giữ riêng; đọc registry mới không tự chạy lại các stage đã hoàn tất. Trial bị ngắt không được tự tiếp tục như thể đã resume đúng RNG/optimizer.

## 1. Setup Kaggle

Upload ZIP source mới nhất làm Input, bật GPU và Internet. Source/Input chỉ đọc, output dưới /kaggle/working. Local có thể dùng backend bằng Suite(root=thư_mục_lab, device='cpu' hoặc 'cuda'); bootstrap bên dưới dành Kaggle.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, shutil, json, os
import torch
assert torch.cuda.is_available(), 'Bật GPU trong cài đặt notebook trước khi chạy.'
print(torch.__version__, torch.cuda.get_device_name(0))
WORK = Path('/kaggle/working/lab02_standard')
# Có nhiều bộ source thì chọn đường dẫn ở đây; mặc định tự tìm.
SOURCE_ROOT = None
SOURCE_ZIP = None
required = ('src/standard_suite.py', 'config/standard.json', 'tests/test_optuna_dense.py')
if (WORK/'outputs/runs').exists():
    assert all((WORK/p).is_file() for p in required), 'Run đã tồn tại nhưng source thiếu. Không ghi đè source của run.'
    print('Giữ nguyên source và run đã có. Không chạy lại cell TRAINING nếu chỉ xem kết quả.')
else:
    if SOURCE_ROOT is None and SOURCE_ZIP is None:
        roots = sorted({p.parent.parent for p in Path('/kaggle/input').rglob('optuna_dense.py')
                        if p.parent.name == 'src' and all((p.parent.parent/f).is_file() for f in required)})
        archives = sorted(Path('/kaggle/input').rglob('lab02_standard_source.zip'))
        if len(roots) == 1:
            SOURCE_ROOT = roots[0]
        elif not roots and len(archives) == 1:
            SOURCE_ZIP = archives[0]
        else:
            print('Source folders:', roots)
            print('Source archives:', archives)
            raise RuntimeError('Không xác định được một bộ source mới. Mở scr-lab02 trong Input, kiểm tra src/standard_suite.py và config/standard.json; đặt SOURCE_ROOT hoặc SOURCE_ZIP nếu có nhiều bản.')
    if SOURCE_ROOT is not None:
        SOURCE_ROOT = Path(SOURCE_ROOT)
        assert all((SOURCE_ROOT/p).is_file() for p in required), 'Dataset thiếu source Optuna mới; upload ZIP mới.'
        WORK.mkdir(parents=True, exist_ok=True)
        for name in ('src','config','tests'):
            shutil.copytree(SOURCE_ROOT/name, WORK/name, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            names = archive.namelist()
            assert all(p in names for p in required), 'ZIP thiếu source/config/test mới.'
            for member in archive.infolist():
                assert (WORK/member.filename).resolve().is_relative_to(WORK.resolve()), 'Đường dẫn ZIP không hợp lệ'
            WORK.mkdir(parents=True, exist_ok=True)
            archive.extractall(WORK)
assert all((WORK/p).is_file() for p in required)
sys.path.insert(0, str(WORK))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna>=4,<5', 'tensorboard', 'scikit-learn', 'pandas', 'matplotlib', 'pytest'], check=True)
print('Source ready:', WORK)


In [ ]:
import platform, importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink, Markdown
from src.delivery import write_json, file_hash
from src.experiment import seed_all
seed_all(42)
SETUP = WORK/'outputs/setup'
SETUP.mkdir(parents=True, exist_ok=True)
environment = {'python':platform.python_version(),'device':torch.cuda.get_device_name(0),
    'libraries':{name:metadata.version(name) for name in ('torch','torchvision','numpy','pandas','optuna','tensorboard')}}
write_json(SETUP/'environment.json', environment)
display(pd.Series(environment['libraries'], name='version').to_frame())


## 2. CIFAR10 và cấu hình

50.000 ảnh official train chia stratified 45.000/5.000, test official 10.000. Checksum xác minh trước khi dùng. Không dùng test để chọn LR.

In [ ]:
raw = list(Path('/kaggle/input').rglob('cifar-10-batches-py'))
if len(raw) == 1:
    DATA = raw[0].parent
else:
    DATA = WORK/'data'
    from torchvision.datasets import CIFAR10
    CIFAR10(str(DATA), train=True, download=True)
    CIFAR10(str(DATA), train=False, download=True)
os.environ['LAB02_DATA'] = str(DATA)
cfg = json.loads((WORK/'config/standard.json').read_text())
CONFIG = WORK/'config/kaggle_optuna.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
from src.eda import read_raw
pool, labels, test_images, test_labels = read_raw(DATA)
assert len(labels) == 50000 and len(test_labels) == 10000
print('Data:', DATA)
print('Budget:', len(cfg['models'])*cfg['trials_per_model'], 'trials, max', cfg['finetune_epochs'], 'epochs/trial')
del pool, test_images


## 3. EDA và kiểm chứng preprocessing

Giữ EDA đầy đủ của Lab02; feature thủ công chỉ mô tả dữ liệu, không nối vào DenseNet/ResNet. EDA preview cũ có resize riêng; preview actual_model_preprocessing bên dưới mới là transform 224x224 thực dùng.

In [ ]:
from src import eda
EDA = eda.build(WORK, data_dir=DATA)
for name in ('structural_overview.csv','integrity_report.csv','class_distribution.csv','pixel_statistics.csv'):
    print(name)
    display(pd.read_csv(EDA/name))
duplicates = pd.read_csv(EDA/'duplicate_images.csv')
print('Exact duplicate records:', len(duplicates))
with np.load(EDA/'split_indices.npz', allow_pickle=False) as split:
    assert len(np.intersect1d(split['train'], split['val'])) == 0
    display(pd.DataFrame({'split':split.files,'samples':[len(split[k]) for k in split.files]}))


In [ ]:
for name in ('sample_images.png','class_distribution.png','class_mean_images.png','pixel_statistics.png','spatial_statistics.png','class_similarity.png','handcrafted_distributions.png','feature_correlation.png'):
    print(name)
    display(Image(filename=str(EDA/name)))


In [ ]:
from src.data import image_transform
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
raw_dataset = CIFAR10(str(DATA), train=True, download=False)
raw_image, label = raw_dataset[0]
transform = image_transform('densenet121', cfg['image_size'])
tensor = transform(raw_image)
assert tensor.shape == (3,224,224) and torch.isfinite(tensor).all()
from src.models import SPECS
contract = SPECS['densenet121'][1].transforms()
recovered = (tensor*torch.tensor(contract.std)[:,None,None] + torch.tensor(contract.mean)[:,None,None]).clamp(0,1)
fig, axes = plt.subplots(1,2,figsize=(7,3))
axes[0].imshow(raw_image); axes[0].set_title('Raw CIFAR10 RGB 32x32')
axes[1].imshow(recovered.permute(1,2,0)); axes[1].set_title('Actual model input, de-normalized')
for ax in axes: ax.axis('off')
fig.tight_layout(); fig.savefig(SETUP/'actual_model_preprocessing.png',dpi=140); display(fig); plt.close(fig)
write_json(SETUP/'preprocessing.json',{'transform':str(transform),'shape':list(tensor.shape),
    'dtype':str(tensor.dtype),'normalization_mean':contract.mean,'normalization_std':contract.std,'augmentation':False})
print(transform)


## 4. Kiến trúc và ngân sách

Head-only chỉ train classifier. Fine-tune ResNet layer4+fc; DenseNet toàn bộ 16 dense layer block4+norm5+classifier. BatchNorm statistics giữ cố định. Tìm LR log [1e-6,1e-3], weight decay cố định; TPE 2 startup trial và 1 trial thích nghi mỗi model. Ba trial là khảo sát ngân sách nhỏ, không chứng minh tối ưu toàn cục.

In [ ]:
from src.models import build_model,configure_trainable,parameter_counts
scopes=[]
for name in cfg['models']:
    model=build_model(name,pretrained=False)
    (SETUP/f'{name}_architecture.txt').write_text(str(model))
    for stage in ('head','finetune'):
        configure_trainable(model,name,stage,'last_block')
        scopes.append({'model':name,'stage':stage,**parameter_counts(model)})
        pd.DataFrame([{'name':n,'parameters':p.numel(),'trainable':p.requires_grad} for n,p in model.named_parameters()]).to_csv(SETUP/f'{name}_{stage}_parameter_scope.csv',index=False)
    del model
scopes=pd.DataFrame(scopes);display(scopes)
fig,ax=plt.subplots(figsize=(9,4))
labels=scopes.model+' '+scopes.stage
ax.barh(labels,scopes.total-scopes.trainable,label='Frozen')
ax.barh(labels,scopes.trainable,left=scopes.total-scopes.trainable,label='Trainable')
ax.set_xlabel('Parameters');ax.legend();fig.tight_layout();fig.savefig(SETUP/'architecture_visual.png',dpi=130);display(fig);plt.close(fig)
display(pd.Series(cfg).to_frame('configuration'))


## 5. Kiểm thử và tạo/reuse run

Test dùng dữ liệu nhỏ, không train benchmark. Registry tự tái dùng các stage hoàn tất; không resume trial bị ngắt. Có thể đặt EXISTING_RUN thành đường dẫn run được khôi phục cùng config. Không chạy lại phiên đang train song song.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','tests/test_standard_suite.py','tests/test_optuna_dense.py','-q'],cwd=WORK,check=True)
from src.standard_suite import Suite
REGISTRY=WORK/'outputs/selected_standard.json'
EXISTING_RUN=None
if EXISTING_RUN is None and REGISTRY.exists():
    EXISTING_RUN=WORK/json.loads(REGISTRY.read_text())['run']
suite=Suite(WORK,config_path=CONFIG,run_path=EXISTING_RUN,device='cuda',live=True)
RUN=suite.run
write_json(REGISTRY,{'run':str(RUN.relative_to(WORK))})
print('Active run:',RUN)


## 6. ResNet18 head-only

Trích xuất feature train/val ở eval mode một lần rồi train head 10 epoch. Plot cập nhật từng epoch, lưu learning_curves.png và history.csv. Chi phí extraction được ghi riêng, không gọi là thời gian train head.

In [ ]:
try:
    resnet_head=suite.head('resnet18')
finally:
    shutil.make_archive('/kaggle/working/lab02_standard_outputs','zip',WORK,'outputs')
display(pd.read_csv(resnet_head/'history.csv'))
display(Image(filename=str(resnet_head/'learning_curves.png')))


## 7. ResNet18 fine-tune và Optuna

3 trial, LR chung cho layer4+fc, giữ split và cùng checkpoint head cho mọi trial. Sau mỗi epoch hiện loss/accuracy/LR; bảng Optuna cập nhật sau mỗi trial. Chưa đọc test.

In [ ]:
try:
    resnet_selection=suite.search('resnet18')
finally:
    shutil.make_archive('/kaggle/working/lab02_standard_outputs','zip',WORK,'outputs')
display(pd.read_csv(RUN/'resnet18/trials.csv'));print(resnet_selection)


## 8. DenseNet121 head-only

Cùng head protocol, seed, batch và ngân sách như ResNet; classifier 10 lớp trên feature cache mới của DenseNet.

In [ ]:
try:
    dense_head=suite.head('densenet121')
finally:
    shutil.make_archive('/kaggle/working/lab02_standard_outputs','zip',WORK,'outputs')
display(pd.read_csv(dense_head/'history.csv'))
display(Image(filename=str(dense_head/'learning_curves.png')))


## 9. DenseNet121 fine-tune và Optuna

Mở block4+norm5+classifier, các phần trước đóng băng. Không mở block3/transition3. Cùng 3 trial và tiêu chí loss như ResNet.

In [ ]:
try:
    dense_selection=suite.search('densenet121')
finally:
    shutil.make_archive('/kaggle/working/lab02_standard_outputs','zip',WORK,'outputs')
display(pd.read_csv(RUN/'densenet121/trials.csv'));print(dense_selection)


## 10. Bốn điều kiện: test, trực quan và full-test sau load

Lúc này cả hai study đã khóa winner. Với từng model, đánh giá head và fine-tune, load lại bằng model instance độc lập và predict đủ 10.000 test. Confusion chứa số đếm; ROC/PR OvR hai panel mỗi điều kiện. Lưới 8 đúng/8 sai không đại diện accuracy. AP khác trapezoidal PR-AUC.

In [ ]:
results=[]
for name in cfg['models']:
    results.append(suite.evaluate(name))
    for stage in ('head','finetune'):
        folder=RUN/name/'evaluation'/stage
        print(name,stage)
        display(pd.Series(json.loads((folder/'metrics.json').read_text())).to_frame('test metrics'))
        for image_name in ('confusion_matrix.png','roc_pr.png','prediction_examples.png'):
            display(Image(filename=str(folder/image_name)))
        display(pd.read_csv(folder/'classification_report.csv'))
        display(pd.read_csv(folder/'roc_pr_metrics.csv'))
        print('Reload verification:',json.loads((folder/'roundtrip.json').read_text()))
        display(Image(filename=str(folder/'loaded/prediction_examples.png')))
        display(pd.read_csv(folder/'loaded/prediction_examples.csv'))
comparison=pd.concat(results,ignore_index=True)
comparison.to_csv(RUN/'comparison.csv',index=False)
display(comparison)


## 11. So sánh và bàn giao

So head/fine-tune cùng kiến trúc trước; fine-tune có thêm training nên không coi là ablation compute bằng nhau. Giữa hai model dùng cùng protocol nhưng số tham số/tốc độ khác nhau. Một seed và ba trial không đủ khẳng định tối ưu hoặc ưu thế thống kê. Đọc val loss tăng cùng train loss giảm để nhận diện overfit; không mặc định accuracy cao chứng minh không overfit. Không tuyên bố dưới 5 giờ nếu chưa đo.

Giữ original run/log/study/checkpoint local sau tải ZIP; chỉ publish evidence được chọn và đã rà riêng tư. Save Version và tải notebook đã chạy có output riêng bằng giao diện Kaggle.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
labels=comparison.model+' '+comparison.stage
for ax,key in zip(axes,('accuracy','loss')):
    bars=ax.bar(labels,comparison[key]);ax.bar_label(bars,fmt='%.4f');ax.set_title('Test '+key);ax.tick_params(axis='x',rotation=20)
fig.tight_layout();fig.savefig(RUN/'comparison.png',dpi=140);display(fig);plt.close(fig)
write_json(RUN/'status.json',{'status':'completed'})
suite.manifest()
package=shutil.make_archive('/kaggle/working/lab02_standard_outputs','zip',WORK,'outputs')
display(FileLink(package));print(package)
